# TAS analysis

This notebook is used to process annual near-surface air temperature (tas) data for land areas between 60°N and 90°N.
When adding a model, include an entry in `MODEL_CONFIGS` and follow the existing directory naming convention.

In [1]:
from pathlib import Path
import sys

# Find this project from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils" / "common.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project folder containing utils/common.py was not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent
CODE_DIR = PROJECT_ROOT / "utils"
sys.path.insert(0, str(CODE_DIR))

from common import find_project_root, load_ec_land_area, remap_land_area_to_esm, map_annual_to_gwl
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
from tas import annual_regional_tas
import xarray as xr

MODEL_CONFIGS = [
    {"name": "EC-Earth3-ESM-1", "family": "EC", "member": "r1i1p1f1"},
    {"name": "GISS-E2-1-G-CC2", "family": "NASA", "member": "r1i1p1f3"},
    {"name": "UKESM1-2-LL", "family": "UKESM", "member": "r1i1p1f1"},
]
EXPERIMENTS = ("Exp1", "Exp2", "Exp4")
GWL_LEGS = {"Exp1": "ramp-up", "Exp4": "ramp-down-2c"}
LAT_MIN, LAT_MAX = 60.0, 90.0

# Store compact annual/GWL data beside the project, outside the repository; raw monthly data is untouched.
OUTPUT_DIR = PROJECT_ROOT.parent / f"{PROJECT_ROOT.name}_plot_data" / "tas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# Use EC land area for EC, and remap it to the NASA/UKESM grid using each model's Exp1 sample coordinates.
ec_land_area = load_ec_land_area(PROJECT_ROOT)
land_areas = {}
for config in MODEL_CONFIGS:
    model, family = config["name"], config["family"]
    if family == "EC":
        land_areas[model] = ec_land_area
    else:
        sample_folder = PROJECT_ROOT / "Exp1_data" / family / f"Exp1_{family}_tas"
        sample_file = sorted(sample_folder.glob("*.nc"))[0]
        with xr.open_dataset(sample_file) as sample:
            land_areas[model] = remap_land_area_to_esm(
                ec_land_area, sample["lat"].load().values, sample["lon"].load().values
            )


In [3]:
# Write one NetCDF per model with annual Exp1/Exp2/Exp4 values, Exp1/Exp4 GWL series, and the Exp2 full-record mean.
for config in MODEL_CONFIGS:
    model, family = config["name"], config["family"]
    annual = {}
    for experiment in EXPERIMENTS:
        folder = PROJECT_ROOT / f"{experiment}_data" / family / f"{experiment}_{family}_tas"
        files = sorted(folder.glob("*.nc"))
        annual[experiment] = annual_regional_tas(files, land_areas[model], lat_min=LAT_MIN, lat_max=LAT_MAX)
        print(f"{model} {experiment}: {annual[experiment].sizes['year']} years; "
              f"mean={float(annual[experiment].mean('year')):.2f} °C")

    mapped = {
        experiment: map_annual_to_gwl(annual[experiment], model, config["member"], leg, PROJECT_ROOT)
        for experiment, leg in GWL_LEGS.items()
    }
    plot_data = xr.Dataset()
    for experiment, series in annual.items():
        plot_data[f"{experiment.lower()}_annual"] = series.rename({"year": f"year_{experiment.lower()}"})
    for experiment, series in mapped.items():
        rename = {"gwl": f"gwl_{experiment.lower()}"}
        if "year" in series.coords:
            rename["year"] = f"source_year_{experiment.lower()}"
        plot_data[f"{experiment.lower()}_gwl"] = series.rename(rename)
    plot_data["exp2_full_record_mean"] = annual["Exp2"].mean("year")
    plot_data.attrs.update({
        "model": model,
        "dataset_family": family,
        "latitude_band": "60–90°N",
        "description": "Annual tas and GWL series for plotting; raw monthly inputs are not included.",
    })
    filename = model.replace("/", "_").replace(" ", "_") + "_tas_plot_data.nc"
    output_file = OUTPUT_DIR / filename
    plot_data.to_netcdf(output_file)
    print(f"Saved plot data: {output_file}")

EC-Earth3-ESM-1 Exp1: 350 years; mean=-1.90 °C
EC-Earth3-ESM-1 Exp2: 500 years; mean=-4.57 °C
EC-Earth3-ESM-1 Exp4: 120 years; mean=-7.88 °C
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/tas/EC-Earth3-ESM-1_tas_plot_data.nc
GISS-E2-1-G-CC2 Exp1: 290 years; mean=-8.31 °C
GISS-E2-1-G-CC2 Exp2: 300 years; mean=-8.13 °C
GISS-E2-1-G-CC2 Exp4: 97 years; mean=-9.78 °C
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/tas/GISS-E2-1-G-CC2_tas_plot_data.nc
UKESM1-2-LL Exp1: 290 years; mean=-4.98 °C
UKESM1-2-LL Exp2: 501 years; mean=-6.77 °C
UKESM1-2-LL Exp4: 134 years; mean=-9.74 °C
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/tas/UKESM1-2-LL_tas_plot_data.nc
